<a href="https://colab.research.google.com/github/Daniela-Alves2004/data-mining/blob/main/Aula_FeautureSelection_Alzheimer_25_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Bibliotecas
import pandas as pd

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
# Carregamento da base de dados
df = pd.read_csv('alzheimers_disease_data.csv')

In [3]:
# Tipos de dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

In [4]:
df['DoctorInCharge'].head()

,DoctorInCharge
0,XXXConfid
1,XXXConfid
2,XXXConfid
3,XXXConfid
4,XXXConfid


### Pré-processamento
Após umam breve análise:
  * não é necessário aplicar substituições (NaN);
  * Não é necessário tranformações (OHE)

In [5]:
df['DoctorInCharge'].value_counts() # ver valores da coluna

,count
DoctorInCharge,
XXXConfid,2149


In [6]:
# Preparação da base
## Remoção das colunas irrelevantes: [PatientID, DoctorInCharge]

cols_remover = ['PatientID', 'DoctorInCharge']
df = df.drop(columns=cols_remover, axis=1)

In [7]:
df['Diagnosis'].value_counts()

,count
Diagnosis,
0,1389
1,760


In [8]:
# Separação de Treino e Teste
X = df.drop(columns = ['Diagnosis'], axis=1)
y = df['Diagnosis']

# Aplica a separação com train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.30,
    stratify = y
)

In [9]:
X_train.shape, X_test.shape

((1504, 32), (645, 32))

In [10]:
# Aplicar a normalização no treino e teste
## Como todas as colunas são numéricas, n
## Não é necessário filtrar com select_dtypes
numeric_cols = X_train.columns.tolist()

# Normalização
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

In [13]:
# Aplica somente o transform no test
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [14]:
# Avalia o modelo com todas as features
clf = DecisionTreeClassifier(random_state= 42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.78      0.72       417
           1       0.41      0.28      0.33       228

    accuracy                           0.60       645
   macro avg       0.54      0.53      0.52       645
weighted avg       0.57      0.60      0.58       645



In [15]:
# Aplicar a seleção de características filter
## Mutual Information

# Especificar o conjunto minimo de features
tam_f = 10

selector = SelectKBest(
    score_func = mutual_info_classif,
    k = tam_f
)

# Treinar a seleção
selector.fit(X_train, y_train)

SelectKBest(score_func=<function mutual_info_classif at 0x7a9388de09a0>)

In [16]:
selector.get_support()

array([False, False, False, False, False, False, False,  True, False,
        True, False,  True, False, False, False, False, False,  True,
       False, False, False, False,  True,  True,  True,  True,  True,
       False,  True, False, False, False])

In [17]:
# Mostrar as features
selected_features = X_train.columns[selector.get_support()].tolist()

In [18]:
selected_features

['PhysicalActivity',
 'SleepQuality',
 'CardiovascularDisease',
 'DiastolicBP',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL',
 'Disorientation']

In [19]:
X_train[selected_features]

,PhysicalActivity,SleepQuality,CardiovascularDisease,DiastolicBP,MMSE,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Disorientation
1339,-1.464565,0.502368,-0.407298,0.537210,0.675833,-0.312292,-0.515745,-0.430331,-0.115009,-0.42271
1394,0.293434,0.011985,-0.407298,0.989029,-0.870808,0.053154,1.938941,-0.430331,0.193475,-0.42271
469,1.143841,0.786246,-0.407298,0.198346,0.694620,-0.032928,-0.515745,-0.430331,-1.294777,-0.42271
1440,-1.565688,-0.008944,-0.407298,0.819597,-0.731385,0.273153,-0.515745,-0.430331,-0.646317,2.36569
1050,-1.332735,1.025867,-0.407298,-0.705290,1.222946,0.295332,-0.515745,-0.430331,1.197417,-0.42271
...,...,...,...,...,...,...,...,...,...,...
599,-1.471036,0.038019,-0.407298,-1.439495,-0.339936,1.318126,-0.515745,-0.430331,0.558080,-0.42271
1073,1.478983,-0.244789,-0.407298,-1.157109,1.149156,0.247444,1.938941,-0.430331,-0.305365,-0.42271
1697,1.730162,1.158805,-0.407298,-0.140517,1.250709,-0.372345,-0.515745,-0.430331,0.803588,-0.42271
1784,1.040017,-0.178496,-0.407298,-0.818245,-0.602076,-1.029870,-0.515745,-0.430331,-1.274950,-0.42271


In [20]:
# Re-treinar o modelo com as selected_features
clf_filter = DecisionTreeClassifier(random_state = 42)
clf_filter.fit(X_train[selected_features], y_train)
y_pred_filter = clf_filter.predict(X_test[selected_features])

In [21]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred_filter ))

              precision    recall  f1-score   support

           0       0.81      0.04      0.08       417
           1       0.36      0.98      0.53       228

    accuracy                           0.37       645
   macro avg       0.58      0.51      0.30       645
weighted avg       0.65      0.37      0.24       645



In [22]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred ))

              precision    recall  f1-score   support

           0       0.66      0.78      0.72       417
           1       0.41      0.28      0.33       228

    accuracy                           0.60       645
   macro avg       0.54      0.53      0.52       645
weighted avg       0.57      0.60      0.58       645



* Qual o subconjunto mínimo para k (SelectKBest)?
* Mutual Information é a melhor abordagem?